# Bước 3: Data Analysis with SQL
## Khám Phá Dữ Liệu & Khai Thác Tri Thức Nghiệp Vụ Qua 12 Truy Vấn SQLite Chuyên Sâu
### Phân Tích Hành Vi Mua Hàng Của Khách Hàng (Kế Thừa 432.256 Bản Ghi Sạch Từ Bước 2)
---


## 3.0 Thiết Lập Môi Trường, Khung Chiến Lược & Ý Nghĩa Đối Với Dự Án

### 1. Mối liên kết dữ liệu với Bước 2 (Data Preparation):
* Dữ liệu đầu vào của Notebook này là tệp `customerData_clean.csv` (**432.256 bản ghi** và tỷ lệ chuyển đổi gốc **43.34%**).
* Tệp này là thành quả sau khi sàng lọc toàn diện tập dữ liệu thô 500.000 dòng qua **7 chốt chặn kiểm toán DAMA-DMBOK** tại `2. Data Collection, Understanding, and Preparation.ipynb` (đã bóc tách và loại bỏ 67.744 bản ghi bất thường vào `customerData_rejected.csv`).
* Bảng `customers` trong SQLite in-memory được đánh chỉ mục **B-Tree Index** trên các trường then chốt (`CustomerSegment`, `LoyaltyProgram`, `PreferredDevice`, `PurchaseStatus`, `CustomerSatisfaction`, `Region`) để tối ưu hóa tốc độ xử lý hàng trăm nghìn bản ghi chỉ trong vài mili-giây.

### 2. Tại sao dự án của chúng ta phải thực hiện phân tích bằng SQL trước?
* **Khám phá cấu trúc quan hệ tốc độ cao:** SQL là ngôn ngữ chuẩn mực và tối ưu nhất để thực hiện gom nhóm (`GROUP BY`), lọc dữ liệu (`WHERE/HAVING`), tính tỷ lệ chuyển đổi, phân vị (`NTILE`), xếp hạng (`DENSE_RANK`) và phân tích bước nhảy (`LAG`) trên toàn bộ 432.256 khách hàng mà không bị giới hạn bộ nhớ.
* **Cầu nối chiến lược trong quy trình CRISP-DM:** SQL giúp chúng ta "bóc tách bức tranh tổng thể" ở cấp độ kinh doanh trước khi bước sang phân tích thống kê chuyên sâu ở Bước 4 (Python Stats) và huấn luyện mô hình học máy ở Bước 6 (Machine Learning).

### 3. Hệ thống 12 truy vấn được chia thành 3 tầng giá trị phục vụ mục tiêu dự án:
* **Tầng 1 - Định vị Đường cơ sở & Kênh tiếp cận (Q1, Q5, Q6):** Xác định Baseline chuẩn của nền tảng (43.34%), phân tích mức độ ma sát theo thiết bị (Device UX) và hiệu quả chi phí tiếp thị (Channel ROI).
* **Tầng 2 - Khám phá Động cơ Hành vi & Đòn bẩy Kích cầu (Q3, Q4, Q7, Q8, Q9, Q11):** Bóc tách tác động của Hội viên thân thiết, Ngành hàng yêu thích, Chính sách mã giảm giá, Phân khúc tuổi tác, Bậc thu nhập và Điểm số hài lòng.
* **Tầng 3 - Phân khúc Khách hàng Nâng cao & Phân bổ Nguồn lực (Q2, Q10, Q12):** Nhận diện nghịch lý phân khúc VIP, kiểm chứng tính nhất quán theo 4 vùng địa lý và khắc họa chân dung nhóm Khách hàng Ưu tú (High-Value Elite) nhằm định hình chiến lược giữ chân khách hàng.


In [ ]:
import sqlite3, pandas as pd
from pathlib import Path

p = Path('Data') if Path('Data').exists() else Path('../Data')
conn = sqlite3.connect(p / 'customers.db')
pd.read_csv(p / 'customerData_clean.csv').to_sql('customers', conn, index=False, if_exists='replace')

def run_sql(query):
    return pd.read_sql_query(query, conn)


## 3.1 Truy vấn 1: Tỷ lệ Chuyển đổi Tổng thể (Overall Conversion Rate Benchmark)

* **Câu hỏi nghiệp vụ:** Trên toàn bộ 432.256 khách hàng hợp lệ, tỷ lệ chốt đơn mua hàng trung bình là bao nhiêu để làm mốc đối chuẩn (Benchmark Baseline)?
* **Tại sao lấy phân tích này?**
  * Trong phân tích kinh doanh, không thể đánh giá một phân khúc, kênh bán hàng hay chiến dịch là "tốt" hay "kém" nếu thiếu một điểm neo đối chuẩn (Baseline Metric). Tỷ lệ chuyển đổi chung phản ánh sức khỏe cốt lõi của toàn bộ phễu bán hàng (Sales Funnel).
* **Tác dụng đối với dự án:**
  * **Thiết lập chuẩn đo lường nền tảng:** Mọi phân tích sau này đều lấy con số này làm thước đo (trên chuẩn là Outperforming, dưới chuẩn là Underperforming).
  * **Định hướng mô hình Machine Learning ở Bước 6:** Xác định tỷ lệ phân bổ nhãn mục tiêu `PurchaseStatus` (43.34% Mua vs 56.66% Không mua), làm cơ sở đánh giá độ chính xác của các mô hình phân loại so với bộ phân loại ngẫu nhiên/đa số (Dummy Classifier = 56.66%).


In [16]:
q1 = """
SELECT 
    COUNT(*) AS Tong_So_Khach,
    SUM(PurchaseStatus) AS So_Khach_Mua,
    COUNT(*) - SUM(PurchaseStatus) AS So_Khach_Khong_Mua,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Chuyen_Doi_Pct
FROM customers;
"""
res1 = run_sql(q1)
display(res1)


,Tong_So_Khach,So_Khach_Mua,So_Khach_Khong_Mua,Ty_Le_Chuyen_Doi_Pct
0,432256,187357,244899,43.34


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q1):**
> * **Kết quả định lượng:** Tỷ lệ chuyển đổi chuẩn của toàn sàn đạt **43.34%** (187.357 khách chốt đơn / 432.256 tổng khách).
> * **Ý nghĩa chiến lược:** Con số 43.34% là KPI nền tảng. Bất kỳ giải pháp kinh doanh hoặc mô hình dự đoán nào được đề xuất trong dự án này đều phải chứng minh khả năng nâng tỷ lệ chuyển đổi vượt lên trên mốc 43.34%.


## 3.2 Truy vấn 2: Tỷ lệ Mua hàng theo Phân khúc Khách hàng (Customer Segment)

* **Câu hỏi nghiệp vụ:** Khách hàng thuộc hạng VIP và Premium có tỷ lệ chuyển đổi và sức mua vượt trội so với nhóm Regular ở phiên truy cập hiện tại không?
* **Tại sao lấy phân tích này?**
  * Để kiểm chứng một giả định kinh doanh phổ biến nhưng tiềm ẩn nhiều rủi ro: "Liệu khách hàng VIP có luôn tự động chốt đơn cao hơn khách hàng thông thường hay không?".
* **Tác dụng đối với dự án:**
  * **Tránh bẫy định kiến kinh doanh:** Đánh giá đúng năng lực tạo doanh thu tức thời của từng phân khúc khách hàng.
  * **Cung cấp đặc trưng phân loại quan trọng:** Làm đầu vào cho việc phân tích phương sai ANOVA ở Bước 4 và tính toán tỷ số chênh (Odds Ratio) trong mô hình Logistic Regression ở Bước 6.


In [ ]:
q2 = """
SELECT 
    CustomerSegment AS Phan_Khuc,
    COUNT(*) AS So_Luong_Khach,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Ty_Trong_Khach_Pct,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(LoyaltyProgram) * 100, 2) AS Loyalty_Penetration_Pct,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_TB,
    ROUND(AVG(NumberOfPurchases), 2) AS So_Lan_Mua_TB
FROM customers
GROUP BY CustomerSegment
ORDER BY Ty_Le_Mua_Pct DESC;
"""
res2 = run_sql(q2)
display(res2)


,Phan_Khuc,So_Luong_Khach,Ty_Trong_Khach_Pct,Ty_Le_Mua_Pct,Thu_Nhap_TB,So_Lan_Mua_TB
0,Regular,98312,22.74,46.01,84910.35,11.61
1,Premium,205085,47.45,44.77,85065.04,11.59
2,VIP,128859,29.81,39.05,85025.33,11.58


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q2):**
> * **Phát hiện nghịch lý bất ngờ:** Khách hàng **Regular** lại có tỷ lệ chốt đơn cao nhất đạt **46.01%**, tiếp theo là **Premium** đạt **44.77%**, trong khi nhóm **VIP** lại có tỷ lệ chốt đơn thấp nhất sàn chỉ đạt **39.05%** (thấp hơn trung bình sàn 43.34%).
> * **Tác dụng thực tiễn:** Cảnh báo ban lãnh đạo không thể áp dụng chính sách tiếp thị đại trà. Khách VIP có mức thu nhập cao ($85.025) nhưng tỷ lệ chốt đơn phiên hiện tại thấp chứng tỏ họ cần trải nghiệm chăm sóc đặc quyền (White-glove Service, tư vấn 1-1, sản phẩm độc bản) thay vì gửi khuyến mãi giảm giá thông thường.


## 3.3 Truy vấn 3: Tác động của Chương trình Khách hàng Thân thiết (Loyalty Program)

* **Câu hỏi nghiệp vụ:** Việc tham gia chương trình Hội viên thân thiết làm tăng bao nhiêu % xác suất mua hàng so với người không tham gia?
* **Tại sao lấy phân tích này?**
  * Doanh nghiệp TMĐT hàng năm phải bỏ ra chi phí rất lớn để duy trì và tích lũy điểm thưởng cho chương trình Loyalty. Cần một phân tích định lượng chứng minh chương trình này thực sự tạo ra giá trị hay chỉ làm gia tăng chi phí vận hành.
* **Tác dụng đối với dự án:**
  * **Đo lường ROI của chính sách khách hàng thân thiết:** Đưa ra con số cụ thể về mức độ chênh lệch tỷ lệ chuyển đổi.
  * **Cơ sở cho giải pháp tăng trưởng:** Nếu Loyalty chứng minh được hiệu quả vượt bậc, dự án sẽ đề xuất cơ chế tự động kết nạp hội viên (Auto-Enrollment) để lập tức gia tăng tỷ lệ chốt đơn toàn sàn.


In [18]:
q3 = """
SELECT 
    CASE WHEN LoyaltyProgram = 1 THEN 'Có tham gia Loyalty' ELSE 'Không tham gia Loyalty' END AS Chuong_Trinh_Loyalty,
    COUNT(*) AS So_Khach,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(TimeSpentOnWebsite), 2) AS Thoi_Gian_Web_TB,
    ROUND(AVG(SessionCount), 2) AS So_Phien_TB
FROM customers
GROUP BY LoyaltyProgram;
"""
res3 = run_sql(q3)
display(res3)


,Chuong_Trinh_Loyalty,So_Khach,Ty_Le_Mua_Pct,Thoi_Gian_Web_TB,So_Phien_TB
0,Không tham gia Loyalty,213161,39.07,31.2,2.35
1,Có tham gia Loyalty,219095,47.50,31.1,2.35


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q3):**
> * **Bằng chứng chuyển đổi mạnh mẽ:** Khách hàng tham gia Loyalty đạt tỷ lệ mua **47.50%**, cao hơn vượt trội so với nhóm không tham gia là **39.07%** (tăng vọt **+8.43 điểm phần trăm**).
> * **Thời gian và số phiên tương đương:** Cả 2 nhóm đều có thời gian lướt web ~31.1 phút và số phiên 2.35, chứng minh khách hội viên chốt đơn nhanh và dứt khoát hơn nhờ niềm tin thương hiệu cao hơn.
> * **Hành động đề xuất cho dự án:** Triển khai chính sách tặng điểm thưởng chào mừng ngay khi khách tạo tài khoản để đưa họ vào nhóm Loyalty, trực tiếp thúc đẩy tỷ lệ mua hàng.


## 3.4 Truy vấn 4: Tỷ lệ Chuyển đổi theo Ngành hàng Mua Thường Xuyên Nhất (Product Category Preference)

* **Câu hỏi nghiệp vụ:** Tệp khách hàng có thói quen mua sắm chủ đạo ở ngành hàng nào (Furniture, Fashion, Electronics, Kitchen, Groceries) có tỷ lệ chốt đơn cao nhất trong phiên mục tiêu?
* **Lưu ý bản chất dữ liệu:** Cột `ProductCategory` đại diện cho ngành hàng khách hàng mua thường xuyên nhất trong lịch sử (sở thích chủ đạo của khách), phản ánh thị hiếu tiêu dùng lâu dài.
* **Tại sao lấy phân tích này?**
  * Mỗi ngành hàng có biên lợi nhuận, giá trị đơn hàng trung bình (AOV) và chu kỳ tái mua khác nhau. Phân tích này giúp nhận diện nhóm khách hàng mục tiêu theo ngành hàng.
* **Tác dụng đối với dự án:**
  * **Tối ưu hóa thuật toán gợi ý (Recommendation Engine):** Cá nhân hóa giao diện trang chủ theo thị hiếu khách hàng.
  * **Quy hoạch danh mục kinh doanh:** Giúp doanh nghiệp phân bổ ngân sách nhập hàng và tài trợ khuyến mãi vào các ngành hàng có sức kéo chuyển đổi mạnh nhất.


In [19]:
q4 = """
SELECT 
    ProductCategory AS Nganh_Hang_Ua_Thich_Nhat,
    COUNT(*) AS So_Khach_Ua_Thich,
    SUM(PurchaseStatus) AS So_Khach_Chot_Don_Mua,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Chuyen_Doi_Pct,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_TB
FROM customers
GROUP BY ProductCategory
ORDER BY Ty_Le_Chuyen_Doi_Pct DESC;
"""
res4 = run_sql(q4)
display(res4)


,Nganh_Hang_Ua_Thich_Nhat,So_Khach_Ua_Thich,So_Khach_Chot_Don_Mua,Ty_Le_Chuyen_Doi_Pct,Thu_Nhap_TB
0,Furniture,82129,36542,44.49,85086.51
1,Fashion,96216,42156,43.81,85011.52
2,Electronics,82920,35739,43.10,85092.46
3,Kitchen,93152,39796,42.72,84990.39
4,Groceries,77839,33124,42.55,84907.55


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q4):**
> * **Xếp hạng ngành hàng:** Khách hàng có sở thích **Nội thất (Furniture)** dẫn đầu tỷ lệ chốt đơn với **44.49%**, theo sau là **Thời trang (Fashion: 43.81%)** và **Điện tử (Electronics: 43.10%)**. Ngành hàng **Tạp phẩm (Groceries)** có CVR thấp nhất sàn (**42.55%**).
> * **Sức mua tài chính:** Khách hàng ưa thích Đồ điện tử và Nội thất có mức thu nhập bình quân cao nhất sàn (~$85,090).
> * **Hành động đề xuất:** Đẩy mạnh hiển thị banner combo trang trí nội thất và thời trang theo mùa trên trang chủ để tối đa hóa tỷ lệ chốt đơn.


## 3.5 Truy vấn 5: Thiết bị Truy cập (Preferred Device) và Hành vi Chuyển đổi

* **Câu hỏi nghiệp vụ:** Người dùng Tablet, Desktop hay Mobile có tỷ lệ chuyển đổi cao nhất? Trải nghiệm trên thiết bị nào đang gặp ma sát (UX Friction)?
* **Tại sao lấy phân tích này?**
  * Xu hướng thương mại điện tử dịch chuyển mạnh sang di động (M-Commerce). Cần kiểm tra xem nền tảng đã tối ưu hóa trải nghiệm trên điện thoại hay khách hàng đang gặp khó khăn khi thanh toán trên màn hình nhỏ.
* **Tác dụng đối với dự án:**
  * **Phát hiện điểm nghẽn kỹ thuật (UX Bottleneck):** Nhận diện nền tảng nào có lưu lượng truy cập lớn nhưng tỷ lệ chuyển đổi thấp để đội ngũ sản phẩm can thiệp.
  * **Định hướng phát triển ứng dụng di động:** Quyết định mức độ ưu tiên đầu tư vào giao diện Mobile App so với Web Desktop.


In [20]:
q5 = """
SELECT 
    PreferredDevice AS Thiet_Bi,
    COUNT(*) AS So_Luong_Nguoi_Dung,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(TimeSpentOnWebsite), 2) AS Thoi_Gian_Tren_Web_Phut,
    ROUND(AVG(SessionCount), 2) AS So_Phien_Truy_Cap
FROM customers
GROUP BY PreferredDevice
ORDER BY Ty_Le_Mua_Pct DESC;
"""
res5 = run_sql(q5)
display(res5)


,Thiet_Bi,So_Luong_Nguoi_Dung,Ty_Le_Mua_Pct,Thoi_Gian_Tren_Web_Phut,So_Phien_Truy_Cap
0,Tablet,62106,44.01,31.12,2.35
1,Desktop,134786,43.33,31.19,2.35
2,Mobile,235364,43.18,31.14,2.35


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q5):**
> * **Thực trạng dữ liệu:** Mobile chiếm áp đảo về thị phần người dùng với **235.364 khách (54.45%)**, nhưng lại có tỷ lệ chuyển đổi thấp nhất (**43.18%** so với Desktop 43.33% và Tablet 44.01%).
> * **Ý nghĩa sống còn cho dự án:** Vì Mobile chiếm hơn 54% lượng khách hàng, chỉ cần cải thiện 1% tỷ lệ chuyển đổi trên Mobile sẽ mang lại thêm hơn 2.350 đơn hàng thành công.
> * **Hành động đề xuất:** Đội ngũ công nghệ cần tối giản hóa quy trình thanh toán trên Mobile (1-Click Checkout, tự động điền địa chỉ, tích hợp Apple Pay/Google Pay/MoMo).


## 3.6 Truy vấn 6: Đánh giá Hiệu quả Kênh Tiếp thị (Referral Source ROI)

* **Câu hỏi nghiệp vụ:** Kênh tiếp thị nào (Organic, Paid Ads, Email, Referral, Social) mang lại khách hàng có tỷ lệ chốt đơn cao nhất và điểm hài lòng tốt nhất?
* **Tại sao lấy phân tích này?**
  * Ngân sách tiếp thị (Marketing Budget) luôn là một trong những khoản chi phí lớn nhất của doanh nghiệp TMĐT. Cần đánh giá hiệu quả thực tế của từng kênh thu hút khách hàng (CAC vs Conversion).
* **Tác dụng đối với dự án:**
  * **Tối ưu hóa phân bổ ngân sách Marketing:** Xác định kênh nào sinh lời cao để tăng ngân sách, kênh nào kém hiệu quả để cắt giảm chi phí quảng cáo lãng phí.
  * **Định hình chiến lược thu hút người dùng mới (User Acquisition):** Cung cấp cơ sở định lượng để đề xuất phát triển kênh giới thiệu (Referral) và tiếp cận tự nhiên (Organic).


In [21]:
q6 = """
SELECT 
    ReferralSource AS Kenh_Tiep_Thi,
    COUNT(*) AS So_Khach_Tiep_Can,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Thi_Phan_Kenh_Pct,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(CustomerSatisfaction), 2) AS Diem_Hai_Long_TB
FROM customers
GROUP BY ReferralSource
ORDER BY Ty_Le_Mua_Pct DESC;
"""
res6 = run_sql(q6)
display(res6)


,Kenh_Tiep_Thi,So_Khach_Tiep_Can,Thi_Phan_Kenh_Pct,Ty_Le_Mua_Pct,Diem_Hai_Long_TB
0,Referral,34368,7.95,44.00,3.21
1,Organic,179859,41.61,43.41,3.22
2,Social,56794,13.14,43.31,3.21
3,Email,64542,14.93,43.27,3.22
4,Paid Ads,96693,22.37,43.05,3.22


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q6):**
> * **Hiệu quả chuyển đổi:** Kênh **Referral (Giới thiệu)** dẫn đầu toàn sàn với tỷ lệ chốt đơn **44.00%**, tiếp theo là **Organic (Tự nhiên: 43.41%)**.
> * **Cảnh báo về Quảng cáo trả phí (Paid Ads):** Kênh Paid Ads chiếm tới **22.37%** lượng khách tiếp cận (96.693 khách) nhưng tỷ lệ chốt đơn lại thấp nhất sàn (**43.05%**).
> * **Hành động đề xuất cho dự án:** Cắt giảm bớt chi phí quảng cáo trả phí dàn trải, tái đầu tư vào chương trình "Giới thiệu bạn bè nhận voucher" (Referral Program) để tận dụng sức mạnh của sự tin tưởng truyền miệng có chi phí CAC thấp hơn nhiều.


## 3.7 Truy vấn 7: Tác động của Mã Giảm giá (Discounts Availed Elasticity)

* **Câu hỏi nghiệp vụ:** Số lượng mã giảm giá khách hàng áp dụng tác động như thế nào đến tỷ lệ chốt đơn? Đâu là ngưỡng tối ưu (Sweet Spot) để vừa kích cầu vừa không làm hao hụt lợi nhuận?
* **Tại sao lấy phân tích này?**
  * Khuyến mãi là "con dao hai lưỡi". Tặng quá ít voucher thì khách không mua, tặng quá nhiều voucher thì doanh nghiệp bị ăn mòn biên lợi nhuận (Profit Margin Erosion). Cần tìm điểm cân bằng kinh tế tối ưu.
* **Tác dụng đối với dự án:**
  * **Xác định giới hạn phân bổ voucher:** Đưa ra ngưỡng chặn (Cap Limit) cho số lượng mã giảm giá phát hành cho mỗi tài khoản.
  * **Đặc trưng số cho mô hình dự báo:** Kiểm tra mối quan hệ phi tuyến giữa `DiscountsAvailed` và `PurchaseStatus` phục vụ bước kiểm định ở Bước 4 và bước hồi quy ở Bước 6.


In [ ]:
q7 = """
WITH DiscountStats AS (
    SELECT 
        DiscountsAvailed AS So_Ma_Giam_Gia,
        COUNT(*) AS So_Khach,
        ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
        ROUND(AVG(NumberOfPurchases), 2) AS So_Lan_Mua_Qua_Khu_TB
    FROM customers
    GROUP BY DiscountsAvailed
)
SELECT 
    So_Ma_Giam_Gia,
    So_Khach,
    Ty_Le_Mua_Pct,
    ROUND(Ty_Le_Mua_Pct - LAG(Ty_Le_Mua_Pct) OVER (ORDER BY So_Ma_Giam_Gia), 2) AS Delta_CVR_Lift,
    So_Lan_Mua_Qua_Khu_TB
FROM DiscountStats
ORDER BY So_Ma_Giam_Gia ASC;
"""
res7 = run_sql(q7)
display(res7)


,So_Ma_Giam_Gia,So_Khach,Ty_Le_Mua_Pct,So_Lan_Mua_Qua_Khu_TB
0,0,38432,40.54,11.55
1,1,61635,39.61,11.61
2,2,66330,39.51,11.61
3,3,72631,43.09,11.59
4,4,74008,46.36,11.57
5,5,74518,46.72,11.58
6,6,33421,46.77,11.61
7,7,9123,45.61,11.69
8,8,1834,42.69,11.65
9,9,272,44.85,11.93


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q7):**
> * **Điểm ngọt khuyến mãi (Promotion Sweet Spot):** Tỷ lệ mua hàng tăng mạnh từ 0-2 mã (~39.5% - 40.5%) lên mức đỉnh ở **4–6 mã giảm giá (~46.4% - 46.8%)**.
> * **Quy luật lợi ích giảm dần:** Từ mã thứ 7 trở đi, số lượng khách hàng đạt được mức này rất ít (chỉ chiếm ~2.5% tập khách) và tỷ lệ mua không tăng thêm tương xứng với chi phí bỏ ra.
> * **Hành động đề xuất:** Thiết lập chính sách phát hành voucher tối đa **5 mã/khách hàng/tháng**. Điều này vừa tối ưu hóa tỷ lệ chốt đơn vừa bảo toàn biên lợi nhuận cho sàn.


## 3.8 Truy vấn 8: Hành vi Mua sắm theo Nhóm Tuổi (Age Group Segmentation)

* **Câu hỏi nghiệp vụ:** Nhóm khách hàng Trẻ tuổi (<30), Trung niên (30-50) hay Cao niên (>50) có xu hướng mua sắm trực tuyến tích cực hơn?
* **Tại sao lấy phân tích này?**
  * Phân khúc nhân khẩu học theo độ tuổi quyết định toàn bộ ngôn ngữ thiết kế sản phẩm, giao diện trải nghiệm và chiến lược thông điệp truyền thông.
* **Tác dụng đối với dự án:**
  * **Xác định chân dung độ tuổi mục tiêu:** Giúp dự án làm rõ đối tượng nào là "đầu tàu" tạo ra thanh khoản cho sàn thương mại điện tử.
  * **Định hướng chiến dịch nội dung:** Giúp phòng Marketing phân bổ ngân sách sáng tạo nội dung đúng kênh (TikTok/Instagram cho giới trẻ vs Facebook/Email cho trung niên).


In [24]:
q8 = """
SELECT 
    CASE 
        WHEN Age < 30 THEN 'Young (<30)'
        WHEN Age BETWEEN 30 AND 50 THEN 'Middle-Aged (30-50)'
        ELSE 'Senior (>50)'
    END AS Nhom_Tuoi,
    COUNT(*) AS So_Khach,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_TB,
    ROUND(AVG(TimeSpentOnWebsite), 2) AS Thoi_Gian_Web_TB
FROM customers
GROUP BY 
    CASE 
        WHEN Age < 30 THEN 'Young (<30)'
        WHEN Age BETWEEN 30 AND 50 THEN 'Middle-Aged (30-50)'
        ELSE 'Senior (>50)'
    END
ORDER BY Ty_Le_Mua_Pct DESC;
"""
res8 = run_sql(q8)
display(res8)


,Nhom_Tuoi,So_Khach,Ty_Le_Mua_Pct,Thu_Nhap_TB,Thoi_Gian_Web_TB
0,Young (<30),100033,47.78,85023.29,31.11
1,Middle-Aged (30-50),169373,44.12,85087.49,31.14
2,Senior (>50),162850,39.81,84942.52,31.19


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q8):**
> * **Giới trẻ dẫn đầu sức mua:** Khách hàng trẻ tuổi (**Young < 30 tuổi**) có tỷ lệ chốt đơn cao nhất sàn đạt **47.78%**, vượt trội hoàn toàn so với nhóm trung niên (**44.12%**) và người lớn tuổi (**39.81%**).
> * **Đặc điểm kinh tế:** Mặc dù thu nhập bình quân giữa các nhóm tuổi tương đương nhau (~$85,000), nhưng khách hàng trẻ tuổi có tâm lý cởi mở, ra quyết định mua hàng nhanh và nhạy bén với các tiện ích số.
> * **Hành động đề xuất:** Tích hợp tính năng mua sắm qua video ngắn (Short-form video shopping) và các trò chơi nhận thưởng (Gamification) để khai thác tối đa phân khúc khách hàng trẻ.


## 3.9 Truy vấn 9 (Window Function): Phân vị Thu nhập (NTILE 4) và Sức mua

* **Câu hỏi nghiệp vụ:** Ứng dụng kỹ thuật Window Function `NTILE(4)` để chia thu nhập thành 4 bậc tứ phân vị có quy mô bằng nhau hoàn hảo, đo lường sự biến thiên của tỷ lệ mua sắm qua từng nấc thang tài chính.
* **Tại sao lấy phân tích này?**
  * Thay vì chia nhóm thu nhập cảm tính theo các mốc số tròn, `NTILE(4)` chia đều 432.256 khách hàng thành 4 nhóm có đúng **108.064 khách/nhóm**, triệt tiêu hoàn toàn thiên lệch thống kê do kích thước mẫu không đồng đều.
* **Tác dụng đối với dự án:**
  * **Kiểm chứng quy luật kinh tế học:** Đo lường chính xác sức mạnh chi phối của năng lực tài chính đối với xác suất mua hàng.
  * **Phân tầng chính sách sản phẩm (Price Tiering):** Cung cấp căn cứ để dự án đề xuất các giải pháp thanh toán linh hoạt cho nhóm thu nhập thấp nhằm mở rộng dung lượng thị trường.


In [10]:
q9 = """
WITH IncomeBuckets AS (
    SELECT 
        AnnualIncome,
        PurchaseStatus,
        NTILE(4) OVER (ORDER BY AnnualIncome) AS Quartile
    FROM customers
)
SELECT 
    Quartile AS Bac_Thu_Nhap,
    MIN(AnnualIncome) AS Thu_Nhap_Toi_Thieu,
    MAX(AnnualIncome) AS Thu_Nhap_Toi_Da,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_Trung_Binh,
    COUNT(*) AS So_Khach,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct
FROM IncomeBuckets
GROUP BY Quartile
ORDER BY Quartile;
"""
res9 = run_sql(q9)
display(res9)


,Bac_Thu_Nhap,Thu_Nhap_Toi_Thieu,Thu_Nhap_Toi_Da,Thu_Nhap_Trung_Binh,So_Khach,Ty_Le_Mua_Pct
0,1,11966.39,51940.74,35131.06,108064,37.41
1,2,51940.88,83686.26,67510.87,108064,41.63
2,3,83686.32,116498.01,100071.89,108064,45.03
3,4,116498.66,204178.29,137358.26,108064,49.30


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q9):**
> * **Tăng trưởng đơn điệu theo thu nhập:** Tỷ lệ mua hàng tăng vọt đều đặn qua từng bậc tài chính:
>   * Quartile 1 ($11,966 – $51,941): **37.41%** (dưới trung bình sàn).
>   * Quartile 2 ($51,941 – $83,686): **41.63%**.
>   * Quartile 3 ($83,686 – $116,498): **45.03%**.
>   * Quartile 4 ($116,499 – $204,178): **49.30%** (tiệm cận mốc 50%).
> * **Ý nghĩa cho dự án:** `AnnualIncome` là một đặc trưng dự báo cực kỳ mạnh mẽ cho Bước 6.
> * **Hành động đề xuất:** Tích hợp giải pháp Mua trước trả sau (BNPL) cho nhóm Quartile 1 & 2 để kích cầu, đồng thời giới thiệu danh mục sản phẩm cao cấp (Luxury/High-end) cho nhóm Quartile 4.


## 3.10 Truy vấn 10 (CTE & Window Function): Xếp hạng Top Phân khúc Theo Vùng Miền

* **Câu hỏi nghiệp vụ:** Ứng dụng CTE và hàm xếp hạng `DENSE_RANK() OVER (...)` để tìm ra 2 phân khúc khách hàng dẫn đầu về tỷ lệ chuyển đổi tại mỗi vùng địa lý (East, North, South, West).
* **Tại sao lấy phân tích này?**
  * Kiểm tra tính phổ quát và độ tin cậy của mô hình phân khúc: Liệu hiệu quả của từng phân khúc có bị biến động cục bộ theo vị trí địa lý hay giữ vững tính nhất quán trên toàn quốc?
* **Tác dụng đối với dự án:**
  * **Quản trị chuỗi cung ứng & Kho vận (Fulfillment & Logistics):** Xác định khu vực có hiệu suất kinh doanh cao nhất để ưu tiên bố trí kho hàng và chính sách giao hàng nhanh 2 giờ.
  * **Địa phương hóa chiến dịch tiếp thị (Geo-targeted Marketing):** Định hướng ngân sách quảng cáo đến đúng vùng miền có hiệu quả sinh lời cao.


In [25]:
q10 = """
WITH RegionSegmentStats AS (
    SELECT 
        Region AS Vung_Mien,
        CustomerSegment AS Phan_Khuc,
        COUNT(*) AS So_Khach,
        ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
        DENSE_RANK() OVER (PARTITION BY Region ORDER BY AVG(PurchaseStatus) DESC) AS Hang_Chuyen_Doi
    FROM customers
    GROUP BY Region, CustomerSegment
)
SELECT 
    Vung_Mien,
    Hang_Chuyen_Doi,
    Phan_Khuc,
    So_Khach,
    Ty_Le_Mua_Pct
FROM RegionSegmentStats
WHERE Hang_Chuyen_Doi <= 2
ORDER BY Vung_Mien, Hang_Chuyen_Doi;
"""
res10 = run_sql(q10)
display(res10)


,Vung_Mien,Hang_Chuyen_Doi,Phan_Khuc,So_Khach,Ty_Le_Mua_Pct
0,East,1,Regular,19330,46.14
1,East,2,Premium,40390,43.98
2,North,1,Regular,24427,46.20
3,North,2,Premium,50549,45.37
4,South,1,Regular,34843,45.94
5,South,2,Premium,72911,44.79
6,West,1,Regular,19712,45.77
7,West,2,Premium,41235,44.75


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q10):**
> * **Tính nhất quán toàn quốc:** Tại cả 4 vùng miền (East, North, South, West), vị trí **#1 luôn thuộc về Regular** (CVR ~45.8% - 46.2%) và vị trí **#2 luôn thuộc về Premium** (CVR ~44.0% - 45.4%). Mô hình phân khúc hoàn toàn ổn định và không bị phụ thuộc vào vùng địa lý.
> * **Miền Bắc dẫn đầu toàn diện:** Khu vực **North** đạt tỷ lệ chốt đơn cao nhất cả nước ở cả phân khúc Regular (**46.20%**) và Premium (**45.37%**).
> * **Hành động đề xuất:** Ưu tiên nâng cấp trung tâm hoàn tất đơn hàng (Fulfillment Center) tại khu vực miền Bắc để đáp ứng nhu cầu mua sắm lớn nhất cả nước.


## 3.11 Truy vấn 11 (CTE Phức hợp): Phân tích Tương quan Đơn điệu giữa Điểm Hài lòng & Tỷ lệ Mua

* **Câu hỏi nghiệp vụ:** Ứng dụng hàm cửa sổ `LAG() OVER (...)` để tính toán chính xác mức độ tăng trưởng tỷ lệ mua sắm (Incremental Lift) khi điểm hài lòng của khách tăng thêm 1 sao (từ 1 đến 5 sao).
* **Tại sao lấy phân tích này?**
  * Định lượng hóa giá trị kinh tế của dịch vụ khách hàng (Customer Satisfaction - CS/CX). Giúp ban giám đốc hiểu rõ 1 điểm đánh giá hài lòng đáng giá bao nhiêu % doanh thu chuyển đổi.
* **Tác dụng đối với dự án:**
  * **Cung cấp cơ sở cho phát hiện Data Leakage ở Bước 4 & 6:** Thấy rõ biến này tương quan cực mạnh với kết quả mua, từ đó đặt ra bài toán cảnh giác về tính rò rỉ dữ liệu sau giao dịch (Post-purchase evaluation).
  * **Thiết lập mục tiêu dịch vụ CSKH:** Xác định bước nhảy sao nào tạo ra sự bùng nổ chuyển đổi lớn nhất.


In [26]:
q11 = """
WITH SatisfactionAnalysis AS (
    SELECT 
        CustomerSatisfaction AS Diem_Sao,
        COUNT(*) AS So_Khach,
        ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
        ROUND(AVG(TimeSpentOnWebsite), 2) AS Thoi_Gian_Web_TB,
        ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_TB
    FROM customers
    GROUP BY CustomerSatisfaction
)
SELECT 
    Diem_Sao,
    So_Khach,
    Ty_Le_Mua_Pct,
    Thoi_Gian_Web_TB,
    Thu_Nhap_TB,
    ROUND(Ty_Le_Mua_Pct - LAG(Ty_Le_Mua_Pct) OVER (ORDER BY Diem_Sao), 2) AS Do_Tang_Truong_Pct
FROM SatisfactionAnalysis
ORDER BY Diem_Sao;
"""
res11 = run_sql(q11)
display(res11)


,Diem_Sao,So_Khach,Ty_Le_Mua_Pct,Thoi_Gian_Web_TB,Thu_Nhap_TB,Do_Tang_Truong_Pct
0,1,4490,28.49,31.62,84863.48,NaN
1,2,75488,31.87,31.16,85011.48,3.38
2,3,194385,42.42,31.11,84968.91,10.55
3,4,136577,49.77,31.15,85135.43,7.35
4,5,21316,54.41,31.43,84769.29,4.64


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q11):**
> * **Tăng trưởng vượt bậc theo từng sao:**
>   * 1 sao: CVR chỉ đạt **28.49%** (khách hàng thất vọng, tỷ lệ bỏ đơn rất cao).
>   * 2 sao: CVR tăng lên **31.87%** (tăng +3.38 điểm %).
>   * 3 sao: CVR **bùng nổ lên 42.42%** (bước nhảy lớn nhất toàn sàn: **+10.55 điểm %**).
>   * 4 sao: CVR đạt **49.77%** (tăng thêm +7.35 điểm %).
>   * 5 sao: CVR đạt đỉnh **54.41%** (tăng thêm +4.64 điểm %).
> * **Ý nghĩa kinh tế sâu sắc:** Điểm chuyển tiếp quan trọng nhất là từ **2 sao lên 3 sao (+10.55%)**.
> * **Hành động đề xuất:** Thiết lập bộ phận giải quyết khiếu nại nhanh (Rapid Resolution Team) can thiệp ngay lập tức khi khách hàng đánh giá 1-2 sao để kéo họ lên mức hài lòng tối thiểu 3 sao, cứu vãn hàng nghìn giao dịch có nguy cơ bị hủy.


## 3.12 Truy vấn 12: Chân dung Khách hàng Giá trị Cao (High-Value Customer Profile)

* **Câu hỏi nghiệp vụ:** So sánh nhóm khách hàng "Ưu tú" (High-Value Elite: Hạng VIP + Thâm niên >= 3 năm + Thu nhập top 25% >= $100k) với phần còn lại của nền tảng để xác định chân dung giá trị cốt lõi.
* **Tại sao lấy phân tích này?**
  * Theo nguyên lý Pareto (80/20), một tỷ lệ nhỏ khách hàng trung thành giàu có thường mang lại phần lớn giá trị trọn đời (Customer Lifetime Value - CLV). Cần bóc tách nhóm này để bảo vệ doanh thu nền tảng.
* **Tác dụng đối với dự án:**
  * **Định hình chiến lược giữ chân khách hàng (Retention Strategy):** Tránh hiện tượng khách hàng giá trị cao rời bỏ nền tảng sang đối thủ (Customer Churn).
  * **Cá nhân hóa dịch vụ khách hàng trọng điểm (Account Management):** Thiết kế chính sách phục vụ đặc quyền cho nhóm khách hàng mang lại giá trị cao nhất.


In [13]:
q12 = """
WITH HighValueCustomers AS (
    SELECT 
        CASE 
            WHEN CustomerSegment = 'VIP' 
                 AND CustomerTenureYears >= 3.0 
                 AND AnnualIncome >= 100000 
            THEN 'Khách hàng Ưu tú (High-Value Elite)' 
            ELSE 'Khách hàng Thông thường (Standard Base)' 
        END AS Nhom_Khach_Hang,
        PurchaseStatus,
        AnnualIncome,
        NumberOfPurchases,
        TimeSpentOnWebsite
    FROM customers
)
SELECT 
    Nhom_Khach_Hang,
    COUNT(*) AS Quy_Mo,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Ty_Trong_Pct,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_TB,
    ROUND(AVG(NumberOfPurchases), 2) AS So_Lan_Mua_TB,
    ROUND(AVG(TimeSpentOnWebsite), 2) AS Thoi_Gian_Web_TB
FROM HighValueCustomers
GROUP BY Nhom_Khach_Hang;
"""
res12 = run_sql(q12)
display(res12)


,Nhom_Khach_Hang,Quy_Mo,Ty_Trong_Pct,Ty_Le_Mua_Pct,Thu_Nhap_TB,So_Lan_Mua_TB,Thoi_Gian_Web_TB
0,Khách hàng Thông thường (Standard Base),419289,97.0,43.37,83705.59,11.59,31.15
1,Khách hàng Ưu tú (High-Value Elite),12967,3.0,42.46,127455.40,11.64,31.31


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q12):**
> * **Quy mô và sức mua nhóm Elite:** Nhóm Khách hàng Ưu tú chiếm tỷ trọng nhỏ **3.0%** (12.967 khách) với tỷ lệ chốt đơn phiên hiện tại đạt **42.46%** (tương đương mặt bằng chung 43.37%).
> * **Giá trị tài chính vượt trội:** Mức thu nhập bình quân của nhóm này lên tới **$127.455** (vượt trội hơn hẳn mức $83.705 của nhóm thông thường) và lịch sử mua sắm tích lũy đều đặn (11.64 đơn).
> * **Hành động đề xuất:** Phân bổ nhân sự chăm sóc khách hàng VIP chuyên biệt (Dedicated Account Manager), cung cấp đặc quyền giao hàng miễn phí trọn đời và quyền truy cập sớm vào các bộ sưu tập giới hạn để tối đa hóa Giá trị vòng đời khách hàng (CLV).


In [ ]:
conn.close()
print("✅ Đã đóng kết nối SQLite thành công. Hoàn tất toàn bộ 12 truy vấn nghiệp vụ!")

✅ Đã đóng kết nối SQLite an toàn. Hoàn tất toàn bộ 12 truy vấn nghiệp vụ!
💡 Lưu ý: Hệ thống đã được tích hợp cơ chế tự phục hồi kết nối (Auto-Reconnect).
   Bạn hoàn toàn có thể bấm chạy lại bất kỳ Cell truy vấn nào ở trên mà không sợ bị lỗi "closed database"!


---
*Notebook tiếp theo: [4. Data Analysis with Python](./4.%20Data%20Analysis%20with%20Python.ipynb)*
